# Imports

In [2]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from sklearn.cluster import KMeans

In [3]:
PLOT_COLORS = ("#1f77b4", "#ff7f0e", "#2ca02c", "#AA4A44", "#CBC3E3")
FONT_SIZES = {"small": 18, "medium": 24, "large": 28}

matplotlib.rcParams.update(
    {
        "figure.figsize": (16, 10),
        "figure.titlesize": FONT_SIZES["large"],
        "font.size": FONT_SIZES["small"],
        "axes.labelsize": FONT_SIZES["medium"],
        "axes.titlesize": FONT_SIZES["medium"],
        "xtick.labelsize": FONT_SIZES["small"],
        "ytick.labelsize": FONT_SIZES["small"],
        "legend.fontsize": FONT_SIZES["small"],
    }
)

# Config

In [4]:
@dataclass(frozen=True)
class Config:
    root: Path
    rate_map: dict
    test_plan: dict
    phy_rates: dict
    kernels: tuple
    iterations: int

    @property
    def rates(self) -> tuple:
        return tuple(self.test_plan.keys())

    @property
    def conditions(self) -> list[tuple]:
        return [
            (r, bw, delay, i)
            for r, cases in self.test_plan.items()
            for bw, delay in cases
            for i in range(1, self.iterations + 1)
        ]

    def get_rate_name(self, rate: int) -> str:
        return self.rate_map.get(rate, f"Rate{rate}")

    def base(self, rate, bw, delay, iteration):
        return f"rate{rate}_bw{bw}_delay{delay}_iter{iteration}"

    def dmesg_path(self, kernel, rate, bw, delay, iteration):
        return (
            self.root
            / kernel
            / "dmesg"
            / f"{self.base(rate, bw, delay, iteration)}.txt"
        )

    def raw_path(self, kernel, rate, bw, delay, iteration):
        return (
            self.root / kernel / "raw" / f"{self.base(rate, bw, delay, iteration)}.json"
        )

    def debug_log_path(self, kernel, rate, bw, delay):
        return self.root / kernel / "logs" / f"rate{rate}_bw{bw}_delay{delay}.log"

    def log_path(self, kernel):
        return self.root / kernel / "session.log"


config = Config(
    root=Path("../results/debug_test_50").resolve(),
    rate_map={
        67:  "HT20-SGI-MCS3",
        71:  "HT20-SGI-MCS7",
        199: "HT40-SGI-MCS7",
        214: "HT40-SGI-MCS14",
    },
    phy_rates={
        67:  28.9,   # HT20-SGI-MCS3,  1 stream,  28.9 Mbps nominal
        71:  72.2,   # HT20-SGI-MCS7,  1 stream,  72.2 Mbps nominal
        199: 150.0,  # HT40-SGI-MCS7,  1 stream, 150 Mbps nominal
        214: 270.0,  # HT40-SGI-MCS14, 2 streams, 270 Mbps nominal
    },
    test_plan={
        67:  [("nolim", "nodelay"), ("nolim", "50"), ("12", "nodelay")],
        71:  [("nolim", "nodelay"), ("nolim", "50"), ("30", "nodelay")],
        199: [("nolim", "nodelay"), ("nolim", "50"), ("70", "nodelay")],
        214: [("nolim", "nodelay"), ("nolim", "50"), ("100", "nodelay")],
    },
    # kernels=("default", "tcpaad"),
    kernels=("tcpaad",),
    iterations=5,
)


print("Project root:", config.root, config.root.exists())
print(f"Conditions: {len(config.conditions)}")

Project root: /home/nyrik/temp/experiment/results/debug_test_50 False
Conditions: 60


# Load Debug Results (iperf3)

In [5]:
def parse_timestamp(ts_str):
    ts_str = re.sub(r",(\d+)([+-]\d{2}:\d{2})", r".\1\2", ts_str)
    ts_str = ts_str.replace("T", " ")
    for fmt in ("%Y-%m-%d %H:%M:%S.%f%z", "%Y-%m-%d %H:%M:%S%z"):
        try:
            return datetime.strptime(ts_str, fmt)
        except ValueError:
            continue
    print(f"WARNING: unparseable timestamp: {ts_str}")
    return ts_str


_TS = re.compile(r"^(\S+)\s")

# Common prefix (named groups: sk, rcv_nxt, rcv_wup, rcv_wnd, rcv_ssthresh)
_PFX = (
    r"sk=(?P<sk>\S+) rcv_nxt=(?P<rcv_nxt>\d+) rcv_wup=(?P<rcv_wup>\d+) "
    r"rcv_wnd=(?P<rcv_wnd>\d+) rcv_ssthresh=(?P<rcv_ssthresh>\d+)"
)
_PFX_BAR = _PFX + r" \|"

# ── RECV ──
_RECV_AAD_UPDATE = re.compile(
    r"TCP_AAD RECV "
    + _PFX_BAR
    + r" UPDATE now_us=(?P<now_us>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
    r" iat_curr=(?P<iat_curr>\d+) iat_min=(?P<iat_min>\d+)"
    r" elapsed_us=(?P<elapsed_us>\d+) ato_us=(?P<ato_us>\d+)"
)
_RECV_AAD_INIT = re.compile(
    r"TCP_AAD RECV "
    + _PFX_BAR
    + r" INIT now_us=(?P<now_us>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
)
_RECV_AAD_STALL = re.compile(
    r"TCP_AAD RECV "
    + _PFX_BAR
    + r" STALL now_us=(?P<now_us>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
    r" iat=(?P<iat>\d+) rto=(?P<rto>\d+)"
)
_RECV_AAD_NOISE = re.compile(
    r"TCP_AAD RECV "
    + _PFX_BAR
    + r" NOISE now_us=(?P<now_us>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
    r" iat=(?P<iat>\d+)"
)
_RECV_AAD_IAT_RESET = re.compile(
    r"TCP_AAD RECV "
    + _PFX_BAR
    + r" IAT_RESET now_us=(?P<now_us>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
    r" old_iat_min=(?P<old_iat_min>\d+)"
)

_RECV_DACK_INIT = re.compile(
    r"TCP_DACK RECV "
    + _PFX_BAR
    + r" INIT now=(?P<now>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
)
_RECV_DACK_STALL = re.compile(
    r"TCP_DACK RECV "
    + _PFX_BAR
    + r" STALL now=(?P<now>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
    r" m=(?P<m>-?\d+) rto=(?P<rto>\d+)"
)
_RECV_DACK = re.compile(
    r"TCP_DACK RECV "
    + _PFX_BAR
    + r" (?P<subtype>\w+) now=(?P<now>\d+) seq=(?P<seq>\d+) end_seq=(?P<end_seq>\d+)"
    r" m=(?P<m>-?\d+) ato=(?P<ato>\d+)"
)

# ── ACK_CHK ──
_ACK_CHK_SEND_NOW_AAD = re.compile(
    r"TCP_AAD ACK_CHK "
    + _PFX_BAR
    + r" SEND_NOW two_seg=(?P<two_seg>\d+) quick=(?P<quick>\d+)"
    r" ack_now=(?P<ack_now>\d+) comp_limit=(?P<comp_limit>\d+) dup_ack=(?P<dup_ack>\d+)"
)
_ACK_CHK_DELAYED_AAD = re.compile(r"TCP_AAD ACK_CHK " + _PFX_BAR + r" DELAYED")
_ACK_CHK_DEFERRED_AAD = re.compile(r"TCP_AAD ACK_CHK " + _PFX_BAR + r" DEFERRED")
_ACK_CHK_COMPRESSED_AAD = re.compile(
    r"TCP_AAD ACK_CHK "
    + _PFX_BAR
    + r" COMPRESSED compressed_ack=(?P<compressed_ack>\d+) delay_us=(?P<delay_us>\d+)"
)
_ACK_CHK_SKIP_AAD = re.compile(r"TCP_AAD ACK_CHK " + _PFX_BAR + r" SKIP")

_ACK_CHK_SEND_NOW_DACK = re.compile(
    r"TCP_DACK ACK_CHK "
    + _PFX_BAR
    + r" SEND_NOW two_seg=(?P<two_seg>\d+) quick=(?P<quick>\d+)"
    r" ack_now=(?P<ack_now>\d+) comp_limit=(?P<comp_limit>\d+) dup_ack=(?P<dup_ack>\d+)"
)
_ACK_CHK_DELAYED_DACK = re.compile(r"TCP_DACK ACK_CHK " + _PFX_BAR + r" DELAYED")
_ACK_CHK_DEFERRED_DACK = re.compile(r"TCP_DACK ACK_CHK " + _PFX_BAR + r" DEFERRED")
_ACK_CHK_COMPRESSED_DACK = re.compile(
    r"TCP_DACK ACK_CHK "
    + _PFX_BAR
    + r" COMPRESSED compressed_ack=(?P<compressed_ack>\d+) delay_us=(?P<delay_us>\d+)"
)
_ACK_CHK_SKIP_DACK = re.compile(r"TCP_DACK ACK_CHK " + _PFX_BAR + r" SKIP")

# ── SCHED ──
_SCHED_AAD = re.compile(
    r"TCP_AAD SCHED "
    + _PFX_BAR
    + r" ato_in_us=(?P<ato_in_us>\d+) ato_us=(?P<ato_us>\d+)"
)
_SCHED_DACK = re.compile(
    r"TCP_DACK SCHED "
    + _PFX_BAR
    + r" ato_in=(?P<ato_in>\d+) ato_final=(?P<ato_final>\d+) timeout_ms=(?P<timeout_ms>\d+)"
)
_SCHED_EARLY = re.compile(r"TCP_DACK SCHED " + _PFX_BAR + r" EARLY_SEND")

# ── TIMER_FIRED ──
_TIMER_FIRED_AAD = re.compile(r"TCP_AAD TIMER_FIRED " + _PFX)
_TIMER_FIRED_DACK = re.compile(r"TCP_DACK TIMER_FIRED " + _PFX)

# ── TIMER_ACK ──
_TIMER_ACK_AAD = re.compile(r"TCP_AAD TIMER_ACK " + _PFX)
_TIMER_ACK_DACK = re.compile(r"TCP_DACK TIMER_ACK " + _PFX)

# ── ACK_SENT ──
_ACK_SENT_AAD = re.compile(r"TCP_AAD ACK_SENT " + _PFX)
_ACK_SENT_DACK = re.compile(r"TCP_DACK ACK_SENT " + _PFX)

# ── CLEANUP ──
_CLEANUP_AAD = re.compile(r"TCP_AAD CLEANUP " + _PFX_BAR + r" reason=(?P<reason>\S+)")
_CLEANUP_DACK = re.compile(
    r"TCP_DACK CLEANUP " + _PFX_BAR + r" bytes=(?P<bytes>\d+) mss=(?P<mss>\d+)"
)

# ── Dispatch table ──

EVENTS = ("recv", "ack_chk", "sched", "timer_fired", "timer_ack", "ack_sent", "cleanup")
_SEND_NOW_FLAGS = ("two_seg", "quick", "ack_now", "comp_limit", "dup_ack")
_KEYWORDS = (
    " RECV ",
    " ACK_CHK ",
    " CLEANUP ",
    " SCHED ",
    " TIMER_ACK ",
    " TIMER_FIRED ",
    " ACK_SENT ",
)

# (regex, event_name, static_extra_fields)
_DISPATCH = {
    ("tcpaad", " RECV "): [
        (_RECV_AAD_UPDATE, "recv", {"subtype": "UPDATE"}),
        (_RECV_AAD_INIT, "recv", {"subtype": "INIT"}),
        (_RECV_AAD_STALL, "recv", {"subtype": "STALL"}),
        (_RECV_AAD_IAT_RESET, "recv", {"subtype": "IAT_RESET"}),
        (_RECV_AAD_NOISE, "recv", {"subtype": "NOISE"}),
    ],
    ("default", " RECV "): [
        (_RECV_DACK_INIT, "recv", {"subtype": "INIT"}),
        (_RECV_DACK_STALL, "recv", {"subtype": "STALL"}),
        (_RECV_DACK, "recv", {}),  # subtype captured by regex
    ],
    ("tcpaad", " ACK_CHK "): [
        (_ACK_CHK_SEND_NOW_AAD, "ack_chk", {"outcome": "SEND_NOW"}),
        (_ACK_CHK_DELAYED_AAD, "ack_chk", {"outcome": "DELAYED"}),
        (_ACK_CHK_DEFERRED_AAD, "ack_chk", {"outcome": "DEFERRED"}),
        (_ACK_CHK_COMPRESSED_AAD, "ack_chk", {"outcome": "COMPRESSED"}),
        (_ACK_CHK_SKIP_AAD, "ack_chk", {"outcome": "SKIP"}),
    ],
    ("default", " ACK_CHK "): [
        (_ACK_CHK_SEND_NOW_DACK, "ack_chk", {"outcome": "SEND_NOW"}),
        (_ACK_CHK_DELAYED_DACK, "ack_chk", {"outcome": "DELAYED"}),
        (_ACK_CHK_DEFERRED_DACK, "ack_chk", {"outcome": "DEFERRED"}),
        (_ACK_CHK_COMPRESSED_DACK, "ack_chk", {"outcome": "COMPRESSED"}),
        (_ACK_CHK_SKIP_DACK, "ack_chk", {"outcome": "SKIP"}),
    ],
    ("tcpaad", " CLEANUP "): [(_CLEANUP_AAD, "cleanup", {})],
    ("default", " CLEANUP "): [(_CLEANUP_DACK, "cleanup", {})],
    ("tcpaad", " SCHED "): [(_SCHED_AAD, "sched", {"subtype": "NORMAL"})],
    ("default", " SCHED "): [
        (_SCHED_DACK, "sched", {"subtype": "NORMAL"}),
        (_SCHED_EARLY, "sched", {"subtype": "EARLY_SEND"}),
    ],
    ("tcpaad", " TIMER_ACK "): [(_TIMER_ACK_AAD, "timer_ack", {})],
    ("default", " TIMER_ACK "): [(_TIMER_ACK_DACK, "timer_ack", {})],
    ("tcpaad", " TIMER_FIRED "): [(_TIMER_FIRED_AAD, "timer_fired", {})],
    ("default", " TIMER_FIRED "): [(_TIMER_FIRED_DACK, "timer_fired", {})],
    ("tcpaad", " ACK_SENT "): [(_ACK_SENT_AAD, "ack_sent", {})],
    ("default", " ACK_SENT "): [(_ACK_SENT_DACK, "ack_sent", {})],
}


def _extract(m, ctx, ts):
    """Extract named groups from regex match, auto-converting numeric strings to int."""
    d = {**ctx, "ts": ts}
    for k, v in m.groupdict().items():
        if v is None:
            continue
        d[k] = int(v) if v.lstrip("-").isdigit() else v.strip()
    return d


def parse_file(path, kernel, rate, bw, delay, iteration):
    rows = {ev: [] for ev in EVENTS}
    ctx = dict(kernel=kernel, rate=rate, bw=bw, delay=delay, iteration=iteration)
    prefix = "TCP_AAD" if kernel == "tcpaad" else "TCP_DACK"

    with open(path) as f:
        for raw in f:
            line = raw.strip()
            if prefix not in line:
                continue
            ts_m = _TS.match(line)
            if not ts_m:
                continue
            ts = parse_timestamp(ts_m.group(1))

            for kw in _KEYWORDS:
                if kw not in line:
                    continue
                for rx, ev, extra in _DISPATCH.get((kernel, kw), []):
                    m = rx.search(line)
                    if m:
                        row = _extract(m, ctx, ts)
                        row.update(extra)
                        if row.get("outcome") == "SEND_NOW":
                            for flag in _SEND_NOW_FLAGS:
                                row[flag] = bool(row.get(flag, 0))
                        rows[ev].append(row)
                        break
                break
    return rows

In [6]:
def _parse_one(args, config):
    kernel, rate, bw, delay, iteration = args
    path = config.dmesg_path(kernel, rate, bw, delay, iteration)
    if not path.exists():
        return kernel, None, str(path)
    return kernel, parse_file(path, kernel, rate, bw, delay, iteration), None


all_rows = {k: {ev: [] for ev in EVENTS} for k in config.kernels}
missing = []

jobs = [
    (kernel, rate, bw, delay, it)
    for kernel in config.kernels
    for rate, bw, delay, it in config.conditions
]

with ProcessPoolExecutor() as pool:
    results = pool.map(partial(_parse_one, config=config), jobs)

for kernel, file_rows, miss in results:
    if miss:
        missing.append(miss)
        continue
    for ev in EVENTS:
        all_rows[kernel][ev].extend(file_rows[ev])

dfs = {
    kernel: {ev: pd.DataFrame(all_rows[kernel][ev]) for ev in EVENTS}
    for kernel in config.kernels
}

if missing:
    print(f"MISSING: {len(missing)} files")
for kernel in config.kernels:
    print(f"\n{kernel}:")
    for ev in EVENTS:
        print(f"  {ev}: {len(dfs[kernel][ev])} rows")


MISSING: 60 files

tcpaad:
  recv: 0 rows
  ack_chk: 0 rows
  sched: 0 rows
  timer_fired: 0 rows
  timer_ack: 0 rows
  ack_sent: 0 rows
  cleanup: 0 rows


# Filtering IPERF3 socket

In [7]:
# df = dfs["tcpaad"]["recv"]
# for r, b, d, i in config.conditions:
#     sel = df[
#         (df["rate"] == r)
#         & (df["bw"] == b)
#         & (df["delay"] == d)
#         & (df["iteration"] == i)
#     ]
#     if sel.empty:
#         print(f"Empty: rate={r} bw={b} delay={d} iter={i}")
#         continue
#     summary = (
#         sel.groupby("sk")["now_us"]
#         .agg(["min", "max", "count"])
#         .assign(duration_us=lambda x: x["max"] - x["min"])
#     )
#     print(f"rate={r} bw={b} delay={d} iter={i}")
#     print(summary, "\n")

In [8]:
filtered = {}

for kernel in dfs:
    filtered[kernel] = {}
    for ev in EVENTS:
        df = dfs[kernel][ev]
        if df.empty or "sk" not in df.columns:
            filtered[kernel][ev] = df
            continue
        parts = []
        for _, grp in df.groupby(["rate", "bw", "delay", "iteration"]):
            main_sk = grp["sk"].value_counts().idxmax()
            parts.append(grp[grp["sk"] == main_sk])
        filtered[kernel][ev] = (
            pd.concat(parts, ignore_index=True) if parts else df.iloc[:0]
        )
dfs = filtered

# All conditions

In [9]:
def classify_bursts(s, rate):
    log_iat = np.log10(s["iat_curr"].values.reshape(-1, 1).astype(float))
    km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(log_iat)
    centers = sorted(10 ** km.cluster_centers_.flatten())
    log_boundary = (np.log10(centers[0]) + np.log10(centers[1])) / 2
    threshold = 10**log_boundary
    s = s.copy()
    s["burst_label"] = np.where(s["iat_curr"] < threshold, "intra", "inter")
    return s, centers, threshold
    
for rate, bw, delay, iteration in config.conditions:
    s = (
        dfs["tcpaad"]["recv"][
            (dfs["tcpaad"]["recv"]["rate"] == rate)
            & (dfs["tcpaad"]["recv"]["bw"] == bw)
            & (dfs["tcpaad"]["recv"]["delay"] == delay)
            & (dfs["tcpaad"]["recv"]["iteration"] == iteration)
            & (dfs["tcpaad"]["recv"]["subtype"] == "UPDATE")
        ]
        .copy()
        .sort_values("now_us")
        .reset_index(drop=True)
    )

    if len(s) < 10:
        print(
            f"SKIP {config.get_rate_name(rate)} bw={bw} delay={delay} iter={iteration}: too few events"
        )
        continue

    s, centers, threshold = classify_bursts(s, rate)
    intra = s[s["burst_label"] == "intra"]
    inter = s[s["burst_label"] == "inter"]
    mss = int(s["end_seq"].sub(s["seq"]).mode()[0])
    theoretical_us = mss * 8 / config.phy_rates[rate]

    print(f"\n{config.get_rate_name(rate)} | bw={bw} delay={delay} iter={iteration}")
    print(
        f"  MSS: {mss} B  theoretical intra gap: {theoretical_us:.1f} μs  ({config.phy_rates[rate]} Mbps)"
    )
    print(f"  Burst threshold:       {threshold:.1f} μs")
    print(
        f"  Intra: {len(intra):>6d} ({100 * len(intra) / len(s):.1f}%)  median={intra['iat_curr'].median():.1f} μs  max={intra['iat_curr'].max():.1f} μs"
    )
    print(
        f"  Inter: {len(inter):>6d} ({100 * len(inter) / len(s):.1f}%)  median={inter['iat_curr'].median():.1f} μs  min={inter['iat_curr'].min():.1f} μs"
    )
    #
    # bins = np.logspace(np.log10(s["iat_curr"].min()), np.log10(s["iat_curr"].max()), 80)
    # fig, ax = plt.subplots(figsize=(10, 4))
    # ax.hist(intra["iat_curr"], bins=bins, alpha=0.7, label="intra-burst")
    # ax.hist(inter["iat_curr"], bins=bins, alpha=0.7, label="inter-burst")
    # ax.axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.0f} μs")
    # ax.axvline(theoretical_us, color="green", linestyle=":", label=f"theoretical={theoretical_us:.0f} μs")
    # ax.axvline(centers[0], color="blue", linestyle=":", label=f"intra center={centers[0]:.0f} μs")
    # ax.axvline(centers[1], color="orange", linestyle=":", label=f"inter center={centers[1]:.0f} μs")
    # ax.set_xscale("log")
    # ax.set_xlabel("iat_curr (μs, log scale)")
    # ax.set_ylabel("Count")
    # ax.set_title(f"IAT classification | {config.get_rate_name(rate)} bw={bw} delay={delay} iter={iteration}")
    # ax.legend()
    # plt.tight_layout()
    # plt.show()

KeyError: 'rate'

In [ ]:
# NOISE_US = 5  # IAT values below this (µs) are ARQ artifacts
# RESET_US_LIST = [1_000_000, 2_000_000]  # reset periods: 0.5s, 1s, 2s
# ALPHAS = [2.0, 3.0, 5.0, 10.0, 15.0]
# EWMA_DECAYS = [0.5, 0.7, 0.9]
# OK_BOUND = 3  # ATO < OK_BOUND * inter_gap counts as "ok"
# ASYM_BACKOFFS = [1.01, 1.1, 1.5, 2.0]  # timer-fire ewma multipliers
# ASYM_CAP_US = 500_000  # 500 ms upper bound on ewma_asym

In [ ]:
# def score_vec(ato, max_intra, inter_gap):
#     """
#     Vectorized burst-boundary classifier.

#     Returns a Series of strings: "short", "hit", "ok", or "long".

#       short : ATO ≤ max_intra                       — timer fires mid-burst
#       hit   : max_intra < ATO < inter_gap            — timer fires in the gap (ideal)
#       ok    : inter_gap ≤ ATO < OK_BOUND*inter_gap   — overshoots by ≤ 2 AMPDUs
#       long  : ATO ≥ OK_BOUND * inter_gap             — too late
#     """
#     return pd.Series(
#         np.select(
#             [ato <= max_intra, ato < inter_gap, ato < OK_BOUND * inter_gap],
#             ["short", "hit", "ok"],
#             default="long",
#         ),
#         index=ato.index,
#     )


# def hit_table(df, groupby_cols):
#     """
#     Aggregate scored burst-boundary rows into a summary table.

#     Columns: groupby_cols + short, hit, ok, long, total, hit_rate, good_rate.

#       hit_rate  = hit / total             — strict: only perfect gaps
#       good_rate = (hit + ok) / total      — relaxed: spans ≤ 2 AMPDUs accepted
#     """
#     g = df.groupby(groupby_cols)["outcome"].value_counts().unstack(fill_value=0)
#     for col in ("short", "hit", "ok", "long"):
#         if col not in g.columns:
#             g[col] = 0
#     g["total"] = g[["short", "hit", "ok", "long"]].sum(axis=1)
#     g["hit_rate"] = g["hit"] / g["total"]
#     g["good_rate"] = (g["hit"] + g["ok"]) / g["total"]
#     return g.reset_index()


# def score_formula(raw, ato_series, alpha, formula_name, condition):
#     """
#     Score a formula at every burst boundary in raw.

#     raw         : DataFrame with burst_label, burst_id, iat_valid, iat_curr
#     ato_series  : Series aligned to raw.index — un-scaled ATO estimate per event
#     alpha       : float multiplier applied to ato_series
#     formula_name: str label written into the result
#     condition   : (rate, bw, delay, iteration)

#     Returns a DataFrame, one row per burst boundary.
#     """
#     df = raw.copy()
#     df["_ato"] = ato_series * alpha

#     # Exclude noise events from both ATO computation and max_intra tracking.
#     valid = df[df["iat_valid"].notna()]
#     intra = valid[valid["burst_label"] == "intra"]
#     inter = valid[valid["burst_label"] == "inter"]

#     # At each burst boundary:
#     #   last_ato  = ATO computed on the final intra segment (what the timer was armed to)
#     #   max_intra = largest IAT within the burst (lower bound for a valid ATO)
#     #   inter_gap = IAT of the boundary event (upper bound for a hit)
#     last_ato = intra.groupby("burst_id")["_ato"].last()
#     max_intra = intra.groupby("burst_id")["iat_valid"].max()
#     inter_gap = inter.groupby("burst_id")["iat_curr"].first()

#     bounds = pd.DataFrame(
#         {"ato": last_ato, "max_intra": max_intra, "inter_gap": inter_gap}
#     ).dropna()

#     if bounds.empty:
#         return pd.DataFrame()

#     bounds["outcome"] = score_vec(
#         bounds["ato"], bounds["max_intra"], bounds["inter_gap"]
#     )
#     bounds["formula"] = formula_name
#     bounds["alpha"] = alpha
#     rate, bw, delay, iteration = condition
#     bounds["rate"] = rate
#     bounds["bw"] = bw
#     bounds["delay"] = delay
#     bounds["iteration"] = iteration

#     return bounds.reset_index()

In [ ]:
# # Build all_seqs: one preprocessed DataFrame per (rate, bw, delay, iteration).
# # Added columns:
# #   iat_valid        — iat_curr with noise-gated values replaced by NaN
# #   iat_min_noreset  — expanding minimum of iat_valid (global floor, never forgets)
# #   iat_min_reset    — cumulative minimum within each 1-second time bucket
# #   ewma_{d}         — EWMA of iat_valid with decay d  (pandas alpha = 1 − d)
# #   burst_id         — each group of intra events + its terminating inter event
# #                      shares the same integer ID

# all_seqs = {}

# for rate, bw, delay, iteration in config.conditions:
#     raw = (
#         dfs["tcpaad"]["recv"][
#             (dfs["tcpaad"]["recv"]["rate"] == rate)
#             & (dfs["tcpaad"]["recv"]["bw"] == bw)
#             & (dfs["tcpaad"]["recv"]["delay"] == delay)
#             & (dfs["tcpaad"]["recv"]["iteration"] == iteration)
#             & (dfs["tcpaad"]["recv"]["subtype"] == "UPDATE")
#         ]
#         .copy()
#         .sort_values("now_us")
#         .reset_index(drop=True)
#     )

#     if len(raw) < 10:
#         continue

#     try:
#         raw, _, _ = classify_bursts(raw, rate)
#     except Exception:
#         continue

#     # Noise gate: sub-threshold IATs are ARQ buffering artifacts.
#     raw["iat_valid"] = raw["iat_curr"].where(raw["iat_curr"] >= NOISE_US)

#     # # Global running minimum — Seytnazarov's original iat_min, no reset.
#     # raw["iat_min_noreset"] = raw["iat_valid"].expanding().min()

#     # Reset running minimum: cummin restarts at each boundary.
#     # Sweeps 0.5s, 1s, 2s to test sensitivity to reset period.
#     for r_us in RESET_US_LIST:
#         bucket = (raw["now_us"] // r_us).astype(int)
#         raw[f"iat_min_reset_{r_us}"] = raw.groupby(bucket)["iat_valid"].transform(
#             lambda x: x.expanding().min()
#         )

#     # # EWMA variants: ignore_na skips noise-gated NaNs without restarting.
#     # for d in EWMA_DECAYS:
#     #     raw[f"ewma_{d}"] = (
#     #         raw["iat_valid"].ewm(alpha=(1.0 - d), adjust=False, ignore_na=True).mean()
#     #     )

#     # EWMA intra-only: feed only intra-burst samples to test whether
#     # inter-burst spikes are the cause of ewma_d underperformance.
#     for d in EWMA_DECAYS:
#         intra_only = raw["iat_valid"].where(raw["burst_label"] == "intra")
#         raw[f"ewma_intra_{d}"] = intra_only.ewm(
#             alpha=(1.0 - d), adjust=False, ignore_na=True
#         ).mean()

#     # Asymmetric EWMA with timer-fire backoff.
#     # Updates downward when iat_curr < current estimate (intra-burst gate).
#     # Doubles on timer fire to allow upward recovery when rate drops.
#     ta_mask = (
#         (dfs["tcpaad"]["timer_ack"]["rate"] == rate)
#         & (dfs["tcpaad"]["timer_ack"]["bw"] == bw)
#         & (dfs["tcpaad"]["timer_ack"]["delay"] == delay)
#         & (dfs["tcpaad"]["timer_ack"]["iteration"] == iteration)
#     )
#     ta_ts = dfs["tcpaad"]["timer_ack"].loc[ta_mask, "ts"].tolist()

#     recv_tuples = list(
#         zip(
#             raw["ts"].tolist(),
#             [False] * len(raw),
#             range(len(raw)),
#             raw["iat_valid"].tolist(),
#         )
#     )
#     timer_tuples = [(ts, True, -1, float("nan")) for ts in ta_ts]
#     timeline = sorted(recv_tuples + timer_tuples, key=lambda x: x[0])

#     for d in EWMA_DECAYS:
#         alpha_ema = 1.0 - d
#         for backoff in ASYM_BACKOFFS:
#             ewma = float("nan")
#             result = [float("nan")] * len(raw)

#             for ts, is_timer, pos, iat in timeline:
#                 if is_timer:
#                     if ewma == ewma:  # not nan
#                         ewma = min(ewma * backoff, ASYM_CAP_US)
#                 else:
#                     if iat == iat:  # iat_valid is not nan
#                         if ewma != ewma:  # first valid sample
#                             ewma = iat
#                         elif iat < ewma:
#                             ewma = d * ewma + alpha_ema * iat
#                         # else: inter-burst spike, skip
#                     result[pos] = ewma

#             raw[f"ewma_asym_{d}_{backoff}"] = result

#     # burst_id: increment after each inter event, so intra events + the
#     # following inter event all share one ID.
#     raw["burst_id"] = (
#         (raw["burst_label"] == "inter").shift(1, fill_value=False).cumsum()
#     )

#     all_seqs[(rate, bw, delay, iteration)] = raw

# print(f"Preprocessed {len(all_seqs)} sequences.")

Preprocessed 60 sequences.


In [ ]:
# # --- Reference: real kernel ATO and ideal ATO

# # real  — ato_us as logged by the kernel (Albert's formula, alpha ~1.5)
# # ideal — geometric mean of (max_intra, inter_gap); always scores as hit

# ref_rows = []

# for cond, raw in all_seqs.items():
#     rate, bw, delay, iteration = cond

#     valid = raw[raw["iat_valid"].notna()]
#     intra = valid[valid["burst_label"] == "intra"]
#     inter = valid[valid["burst_label"] == "inter"]

#     max_intra = intra.groupby("burst_id")["iat_valid"].max()
#     inter_gap = inter.groupby("burst_id")["iat_curr"].first()
#     last_real = intra.groupby("burst_id")["ato_us"].last()

#     bounds = pd.DataFrame(
#         {"max_intra": max_intra, "inter_gap": inter_gap, "real_ato": last_real}
#     ).dropna()
#     if bounds.empty:
#         continue

#     for label, ato_col in [("real", "real_ato"), ("ideal", None)]:
#         b = bounds.copy()
#         b["ato"] = b[ato_col] if ato_col else np.sqrt(b["max_intra"] * b["inter_gap"])
#         b["outcome"] = score_vec(b["ato"], b["max_intra"], b["inter_gap"])
#         b["formula"] = label
#         b["alpha"] = float("nan")
#         b["rate"] = rate
#         b["bw"] = bw
#         b["delay"] = delay
#         b["iteration"] = iteration
#         ref_rows.append(b.reset_index())

# df_ref = pd.concat(ref_rows, ignore_index=True)
# print("=== Reference: real vs ideal ===")
# print(hit_table(df_ref, ["formula", "rate", "bw", "delay"]).to_string(index=False))

=== Reference: real vs ideal ===
formula  rate    bw   delay   hit  long    ok  short  total  hit_rate  good_rate
  ideal    67    12 nodelay 13821     0     0      0  13821  1.000000   1.000000
  ideal    67 nolim      50  4676     0     0      0   4676  1.000000   1.000000
  ideal    67 nolim nodelay  9232     0     0      0   9232  1.000000   1.000000
  ideal    71    30 nodelay 52231     0     0      0  52231  1.000000   1.000000
  ideal    71 nolim      50  7026     0     0      0   7026  1.000000   1.000000
  ideal    71 nolim nodelay  8200     0     0      0   8200  1.000000   1.000000
  ideal   199    70 nodelay 45613     0     0      0  45613  1.000000   1.000000
  ideal   199 nolim      50  3788     0     0      0   3788  1.000000   1.000000
  ideal   199 nolim nodelay 11197     0     0      0  11197  1.000000   1.000000
  ideal   214   100 nodelay 54491     0     0      0  54491  1.000000   1.000000
  ideal   214 nolim      50  6105     0     0      0   6105  1.000000   1.00

In [ ]:
# # --- Formula A_reset: ato = iat_min_reset * alpha

# # iat_min floor resets to NaN at each 1-second boundary and rebuilds from the
# # next arriving segment. After the reset, the first post-reset sample immediately
# # sets iat_min = iat_curr (no U64_MAX spike), so the reset does not corrupt ato_us.

# rows_A_reset = []
# for cond, raw in all_seqs.items():
#     for r_us in RESET_US_LIST:
#         for alpha in ALPHAS:
#             rows_A_reset.append(
#                 score_formula(
#                     raw,
#                     raw[f"iat_min_reset_{r_us}"],
#                     alpha,
#                     f"A_reset_{r_us // 1000}ms",
#                     cond,
#                 )
#             )

# df_A_reset = pd.concat([r for r in rows_A_reset if not r.empty], ignore_index=True)
# print("\n=== Formula A_reset ===")
# print(
#     hit_table(df_A_reset, ["formula", "alpha"])
#     .sort_values(["formula", "good_rate"], ascending=[True, False])
#     .to_string(index=False)
# )


=== Formula A_reset ===
       formula  alpha    hit  long    ok  short  total  hit_rate  good_rate
A_reset_1000ms   10.0 153685 24441 43608  21346 243080  0.632240   0.811638
A_reset_1000ms   15.0 137169 36250 57423  12238 243080  0.564296   0.800527
A_reset_1000ms    5.0 129814   342 35908  77016 243080  0.534038   0.681759
A_reset_1000ms    3.0  84181     0 17805 141094 243080  0.346310   0.419557
A_reset_1000ms    2.0  70492     0   699 171889 243080  0.289995   0.292871
A_reset_2000ms   15.0 152703 25797 44023  20557 243080  0.628201   0.809306
A_reset_2000ms   10.0 162662 17378 29364  33676 243080  0.669171   0.789970
A_reset_2000ms    5.0 110504   213 25584 106779 243080  0.454599   0.559849
A_reset_2000ms    3.0  54634     0 12659 175787 243080  0.224757   0.276835
A_reset_2000ms    2.0  47273     0   435 195372 243080  0.194475   0.196265


In [ ]:
# # --- Formula A_noreset: ato = iat_min_noreset * alpha
#
# # Seytnazarov's original approach: iat_min accumulates the global minimum and
# # never forgets. Converges to the true MPDU transmission time for a stable PHY
# # rate. Adapts slowly when the rate changes.
#
# rows_A_noreset = []
# for cond, raw in all_seqs.items():
#     for alpha in ALPHAS:
#         rows_A_noreset.append(
#             score_formula(raw, raw["iat_min_noreset"], alpha, "A_noreset", cond)
#         )
#
# df_A_noreset = pd.concat([r for r in rows_A_noreset if not r.empty], ignore_index=True)
# print("\n=== Formula A_noreset ===")
# print(hit_table(df_A_noreset, ["alpha"]).sort_values("good_rate", ascending=False).to_string(index=False))

In [ ]:
# # --- Formula B_reset: ato = (0.75 * iat_min_reset + 0.25 * iat_curr) * alpha

# # Albert's published blend: adds iat_curr to make the estimate react faster
# # when a larger MPDU arrives. Weights 0.75 / 0.25 from his thesis.
# # Combined with the 1s reset, this is what the kernel currently runs.

# rows_B_reset = []
# for cond, raw in all_seqs.items():
#     for r_us in RESET_US_LIST:
#         ato_base = 0.75 * raw[f"iat_min_reset_{r_us}"] + 0.25 * raw["iat_valid"]
#         for alpha in ALPHAS:
#             rows_B_reset.append(
#                 score_formula(raw, ato_base, alpha, f"B_reset_{r_us // 1000}ms", cond)
#             )

# df_B_reset = pd.concat([r for r in rows_B_reset if not r.empty], ignore_index=True)
# print("\n=== Formula B_reset ===")
# print(
#     hit_table(df_B_reset, ["formula", "alpha"])
#     .sort_values(["formula", "good_rate"], ascending=[True, False])
#     .to_string(index=False)
# )


=== Formula B_reset ===
       formula  alpha    hit  long    ok  short  total  hit_rate  good_rate
B_reset_1000ms    5.0 179830   377 38409  24464 243080  0.739798   0.897807
B_reset_1000ms   10.0 150459 27618 62645   2358 243080  0.618969   0.876683
B_reset_1000ms   15.0 112162 38786 91989    143 243080  0.461420   0.839851
B_reset_1000ms    3.0 148483     0 21590  73007 243080  0.610840   0.699659
B_reset_1000ms    2.0  84675     0  2226 156179 243080  0.348342   0.357500
B_reset_2000ms   10.0 164500 20052 55273   3255 243080  0.676732   0.904118
B_reset_2000ms   15.0 119511 28020 95371    178 243080  0.491653   0.883997
B_reset_2000ms    5.0 183369   236 27784  31691 243080  0.754357   0.868656
B_reset_2000ms    3.0 137006     0 15761  90313 243080  0.563625   0.628464
B_reset_2000ms    2.0  55833     0  1699 185548 243080  0.229690   0.236679


In [ ]:
# # --- Formula B_noreset: ato = (0.75 * iat_min_noreset + 0.25 * iat_curr) * alpha
#
# # Albert's blend but with the global minimum floor (no reset).
# # Tests whether the 1s reset actually helps or if iat_min_noreset is a better floor.
#
# rows_B_noreset = []
# for cond, raw in all_seqs.items():
#     ato_base = 0.75 * raw["iat_min_noreset"] + 0.25 * raw["iat_valid"]
#     for alpha in ALPHAS:
#         rows_B_noreset.append(
#             score_formula(raw, ato_base, alpha, "B_noreset", cond)
#         )
#
# df_B_noreset = pd.concat([r for r in rows_B_noreset if not r.empty], ignore_index=True)
# print("\n=== Formula B_noreset ===")
# print(hit_table(df_B_noreset, ["alpha"]).sort_values("good_rate", ascending=False).to_string(index=False))

In [ ]:
# # --- Formula C: ato = iat_curr * alpha
#
# # No state at all. Uses only the most recent IAT as the estimate.
# # Simplest possible formula — works if within-burst IATs are stable enough.
# # When the 1s reset fires in Formulas A/B, iat_min temporarily equals iat_curr
# # on the first post-reset packet, so those formulas momentarily degenerate to C.
#
# rows_C = []
# for cond, raw in all_seqs.items():
#     for alpha in ALPHAS:
#         rows_C.append(score_formula(raw, raw["iat_valid"], alpha, "C", cond))
#
# df_C = pd.concat([r for r in rows_C if not r.empty], ignore_index=True)
# print("\n=== Formula C ===")
# print(
#     hit_table(df_C, ["alpha"])
#     .sort_values("good_rate", ascending=False)
#     .to_string(index=False)
# )

In [ ]:
# # --- Formula D: ato = ewma(iat_curr) * alpha
#
# # EWMA smooths burst-to-burst noise without any explicit reset.
# # High decay (0.9) = long memory, risks blending in inter-burst spikes.
# # Low decay (0.5) ≈ Formula C (dominated by the most recent sample).
# # Sweep both decay and alpha to find the best combination.
#
# rows_D = []
# for cond, raw in all_seqs.items():
#     for d in EWMA_DECAYS:
#         for alpha in ALPHAS:
#             rows_D.append(score_formula(raw, raw[f"ewma_{d}"], alpha, f"D_d{d}", cond))
#
# df_D = pd.concat([r for r in rows_D if not r.empty], ignore_index=True)
# print("\n=== Formula D: hit table per decay × alpha ===")
# print(
#     hit_table(df_D, ["formula", "alpha"])
#     .sort_values(["formula", "good_rate"], ascending=[True, False])
#     .to_string(index=False)
# )

In [ ]:
# # --- Formula E: ato = ewma_intra(iat_curr) * alpha
# #
# # EWMA updated only on intra-burst classified samples.
# # Tests whether excluding inter-burst spikes from the EWMA removes the bias
# # that causes Formula D to underperform Formula C.

# rows_E = []
# for cond, raw in all_seqs.items():
#     for d in EWMA_DECAYS:
#         for alpha in ALPHAS:
#             rows_E.append(
#                 score_formula(raw, raw[f"ewma_intra_{d}"], alpha, f"E_d{d}", cond)
#             )

# df_E = pd.concat([r for r in rows_E if not r.empty], ignore_index=True)
# print("\n=== Formula E: EWMA intra-only, hit table per decay × alpha ===")
# print(
#     hit_table(df_E, ["formula", "alpha"])
#     .sort_values(["formula", "good_rate"], ascending=[True, False])
#     .to_string(index=False)
# )


=== Formula E: EWMA intra-only, hit table per decay × alpha ===
formula  alpha    hit  long    ok  short  total  hit_rate  good_rate
 E_d0.5    5.0 157646  5706 79210    518 243080  0.648535   0.974395
 E_d0.5    3.0 195607     0 38414   9059 243080  0.804702   0.962732
 E_d0.5    2.0 199198     0 10077  33805 243080  0.819475   0.860931
 E_d0.5   10.0  99544 45116 98401     19 243080  0.409511   0.814320
 E_d0.5   15.0  75407 84916 82742     15 243080  0.310215   0.650605
 E_d0.7    5.0 155036  5493 82168    383 243080  0.637798   0.975827
 E_d0.7    3.0 195871     0 39041   8168 243080  0.805788   0.966398
 E_d0.7    2.0 200344     0 10000  32736 243080  0.824190   0.865328
 E_d0.7   10.0  98191 47042 97828     19 243080  0.403945   0.806397
 E_d0.7   15.0  74026 87661 81380     13 243080  0.304533   0.639320
 E_d0.9    5.0 151887  5240 85665    288 243080  0.624844   0.977259
 E_d0.9    3.0 194992     0 40256   7832 243080  0.802172   0.967780
 E_d0.9    2.0 201264     0  9832  319

In [ ]:
# # --- Formula F: asymmetric EWMA with timer-fire backoff
# #
# # Updates downward when iat_curr < current estimate (intra-burst gate).
# # Doubles on timer fire to allow upward recovery when rate drops.
# # Mirrors the default kernel's asymmetric ATO update + expiry backoff,
# # at microsecond resolution.

# rows_F = []
# for cond, raw in all_seqs.items():
#     for d in EWMA_DECAYS:
#         for backoff in ASYM_BACKOFFS:
#             for alpha in ALPHAS:
#                 rows_F.append(
#                     score_formula(
#                         raw,
#                         raw[f"ewma_asym_{d}_{backoff}"],
#                         alpha,
#                         f"F_d{d}_b{backoff}",
#                         cond,
#                     )
#                 )

# df_F = pd.concat([r for r in rows_F if not r.empty], ignore_index=True)
# print("\n=== Formula F: asymmetric EWMA with timer-fire backoff ===")
# print(
#     hit_table(df_F, ["formula", "alpha"])
#     .sort_values(["formula", "good_rate"], ascending=[True, False])
#     .to_string(index=False)
# )


=== Formula F: asymmetric EWMA with timer-fire backoff ===
     formula  alpha    hit   long    ok  short  total  hit_rate  good_rate
F_d0.5_b1.01    5.0 164884    145 64783  13268 243080  0.678312   0.944821
F_d0.5_b1.01   10.0 119127  33785 87121   3047 243080  0.490073   0.848478
F_d0.5_b1.01    3.0 172198     20 29327  41535 243080  0.708401   0.829048
F_d0.5_b1.01   15.0  93226  64928 83654   1272 243080  0.383520   0.727662
F_d0.5_b1.01    2.0 156281     14   380  86405 243080  0.642920   0.644483
 F_d0.5_b1.1    5.0 159617   1061 77569   4833 243080  0.656644   0.975753
 F_d0.5_b1.1    3.0 187008     27 35642  20403 243080  0.769327   0.915954
 F_d0.5_b1.1   10.0 104791  40027 97120   1142 243080  0.431097   0.830636
 F_d0.5_b1.1    2.0 186178     14  4452  52436 243080  0.765912   0.784227
 F_d0.5_b1.1   15.0  82669  78630 81420    361 243080  0.340090   0.675041
 F_d0.5_b1.5    5.0 148596  12238 78723   3523 243080  0.611305   0.935161
 F_d0.5_b1.5    3.0 180688   2427 44041 

In [ ]:
# # --- Comparison: best alpha per formula by good_rate

# all_dfs = {
#     "real": df_ref[df_ref["formula"] == "real"],
#     "ideal": df_ref[df_ref["formula"] == "ideal"],
#     **{
#         f"A_reset_{r_us // 1000}ms": df_A_reset[
#             df_A_reset["formula"] == f"A_reset_{r_us // 1000}ms"
#         ]
#         for r_us in RESET_US_LIST
#     },
#     # "A_noreset": df_A_noreset,
#     **{
#         f"B_reset_{r_us // 1000}ms": df_B_reset[
#             df_B_reset["formula"] == f"B_reset_{r_us // 1000}ms"
#         ]
#         for r_us in RESET_US_LIST
#     },
#     # "B_noreset": df_B_noreset,
#     # "C": df_C,
#     # **{f"D_d{d}": df_D[df_D["formula"] == f"D_d{d}"] for d in EWMA_DECAYS},
#     **{f"E_d{d}": df_E[df_E["formula"] == f"E_d{d}"] for d in EWMA_DECAYS},
#     **{
#         f"F_d{d}_b{b}": df_F[df_F["formula"] == f"F_d{d}_b{b}"]
#         for d in EWMA_DECAYS
#         for b in ASYM_BACKOFFS
#     },
# }

# # For each formula, find the alpha with the highest overall good_rate.
# best_rows = []
# for fname, fdf in all_dfs.items():
#     if fdf.empty:
#         continue
#     if fname in ("real", "ideal"):
#         tbl = hit_table(fdf, ["formula"])
#         best_rows.append(tbl.assign(alpha=float("nan")))
#         continue
#     tbl = hit_table(fdf, ["formula", "alpha"])
#     best_rows.append(tbl.sort_values("good_rate", ascending=False).iloc[:1])

# df_comparison = pd.concat(best_rows, ignore_index=True)
# print("\n=== Comparison: best alpha per formula (overall) ===")
# _cols = [
#     "formula",
#     "alpha",
#     "hit",
#     "ok",
#     "short",
#     "long",
#     "total",
#     "hit_rate",
#     "good_rate",
# ]
# print(
#     df_comparison[[c for c in _cols if c in df_comparison.columns]]
#     .sort_values("good_rate", ascending=False)
#     .to_string(index=False)
# )


=== Comparison: best alpha per formula (overall) ===
       formula  alpha    hit    ok  short   long  total  hit_rate  good_rate
         ideal    NaN 243080     0      0      0 243080  1.000000   1.000000
  F_d0.9_b1.01    5.0 160094 79971   2354    661 243080  0.658606   0.987597
        E_d0.9    5.0 151887 85665    288   5240 243080  0.624844   0.977259
        E_d0.7    5.0 155036 82168    383   5493 243080  0.637798   0.975827
   F_d0.5_b1.1    5.0 159617 77569   4833   1061 243080  0.656644   0.975753
        E_d0.5    5.0 157646 79210    518   5706 243080  0.648535   0.974395
   F_d0.7_b1.1    5.0 154276 82331   2900   3573 243080  0.634672   0.973371
  F_d0.7_b1.01    5.0 165087 69961   7790    242 243080  0.679147   0.966957
  F_d0.5_b1.01    5.0 164884 64783  13268    145 243080  0.678312   0.944821
   F_d0.5_b1.5    5.0 148596 78723   3523  12238 243080  0.611305   0.935161
   F_d0.9_b1.1    3.0 163624 58542   8987  11927 243080  0.673128   0.913962
B_reset_2000ms   10.0 

In [ ]:
# print("\n=== Comparison: best alpha per formula (sorted by hit_rate + good_rate) ===")
# _cols_score = _cols + ["score"]
# print(
#     df_comparison.assign(score=(df_comparison["hit_rate"] + df_comparison["good_rate"]) / 2)
#     [[c for c in _cols_score if c in df_comparison.columns or c == "score"]]
#     .sort_values("score", ascending=False)
#     .to_string(index=False)
# )


=== Comparison: best alpha per formula (sorted by hit_rate + good_rate) ===
       formula  alpha    hit    ok  short   long  total  hit_rate  good_rate    score
         ideal    NaN 243080     0      0      0 243080  1.000000   1.000000 1.000000
  F_d0.9_b1.01    5.0 160094 79971   2354    661 243080  0.658606   0.987597 0.823101
  F_d0.7_b1.01    5.0 165087 69961   7790    242 243080  0.679147   0.966957 0.823052
B_reset_1000ms    5.0 179830 38409  24464    377 243080  0.739798   0.897807 0.818802
   F_d0.5_b1.1    5.0 159617 77569   4833   1061 243080  0.656644   0.975753 0.816198
  F_d0.5_b1.01    5.0 164884 64783  13268    145 243080  0.678312   0.944821 0.811566
        E_d0.5    5.0 157646 79210    518   5706 243080  0.648535   0.974395 0.811465
        E_d0.7    5.0 155036 82168    383   5493 243080  0.637798   0.975827 0.806813
   F_d0.7_b1.1    5.0 154276 82331   2900   3573 243080  0.634672   0.973371 0.804021
        E_d0.9    5.0 151887 85665    288   5240 243080  0.6248

In [ ]:
# # Per-condition breakdown at each formula's best global alpha.
# print("\n=== Per-condition breakdown at best alpha ===")
# for fname, fdf in all_dfs.items():
#     if fdf.empty or fname in ("real", "ideal"):
#         continue
#     tbl_global = hit_table(fdf, ["formula", "alpha"])
#     best_alpha = tbl_global.sort_values("good_rate", ascending=False).iloc[0]["alpha"]
#     tbl_cond = hit_table(
#         fdf[fdf["alpha"] == best_alpha], ["formula", "rate", "bw", "delay"]
#     )
#     print(f"\n{fname}  (α={best_alpha})")
#     _c = [
#         "rate",
#         "bw",
#         "delay",
#         "hit",
#         "ok",
#         "short",
#         "long",
#         "total",
#         "hit_rate",
#         "good_rate",
#     ]
#     print(tbl_cond[[c for c in _c if c in tbl_cond.columns]].to_string(index=False))


=== Per-condition breakdown at best alpha ===

A_reset_1000ms  (α=10.0)
 rate    bw   delay   hit    ok  short  long  total  hit_rate  good_rate
   67    12 nodelay   114  3204    596  9907  13821  0.008248   0.240069
   67 nolim      50  1024  1310   1468   874   4676  0.218991   0.499145
   67 nolim nodelay   774  2292   3158  3008   9232  0.083839   0.332106
   71    30 nodelay 17696 23633    263 10639  52231  0.338803   0.791273
   71 nolim      50  3407  1440   2173     6   7026  0.484913   0.689866
   71 nolim nodelay  3666  1366   3167     1   8200  0.447073   0.613659
  199    70 nodelay 34838  8504   2270     1  45613  0.763773   0.950212
  199 nolim      50  2609    19   1158     2   3788  0.688754   0.693770
  199 nolim nodelay  7204    47   3946     0  11197  0.643387   0.647584
  214   100 nodelay 51609  1240   1642     0  54491  0.947111   0.969867
  214 nolim      50  5241   106    755     3   6105  0.858477   0.875839
  214 nolim nodelay 25503   447    750     0  26700

In [ ]:
# _vis_rate, _vis_bw, _vis_delay = SIM_RATE, SIM_BW, SIM_DELAY

# # Pool all iterations for the selected condition to maximise boundary samples.
# seq_vis = pd.concat(
#     [raw for (r, b, d, _), raw in all_seqs.items()
#      if r == _vis_rate and b == _vis_bw and d == _vis_delay],
#     ignore_index=True,
# )

# if seq_vis.empty:
#     print("No data for visualisation condition.")
# else:
#     intra_vis = seq_vis[seq_vis["burst_label"] == "intra"]["iat_curr"]
#     inter_vis = seq_vis[seq_vis["burst_label"] == "inter"]["iat_curr"]

#     _iat_min_v = max(1.0, seq_vis["iat_curr"].min())
#     _iat_max_v = seq_vis["iat_curr"].max()
#     bins = np.logspace(np.log10(_iat_min_v), np.log10(_iat_max_v), 60)

#     # Reference lines (same as the IAT classification plot above).
#     _, _vis_centers, _vis_threshold = classify_bursts(seq_vis, _vis_rate)
#     _vis_mss         = int((seq_vis["end_seq"] - seq_vis["seq"]).mode()[0])
#     _vis_theoretical = _vis_mss * 8 / config.phy_rates[_vis_rate]

#     # Show the seven base formulas plus the best-performing D variant.
#     vis_formulas = ["real", "ideal", "A_reset", "A_noreset", "B_reset", "B_noreset", "C"]
#     if not df_D.empty:
#         _best_D = (
#             hit_table(df_D, ["formula", "alpha"])
#             .sort_values("good_rate", ascending=False)
#             .iloc[0]["formula"]
#         )
#         vis_formulas.append(_best_D)

#     for fname in vis_formulas:
#         fig, ax = plt.subplots()
#         ax.set_title(
#             f"{fname}  |  {config.get_rate_name(_vis_rate)}"
#             f"  bw={_vis_bw}  delay={_vis_delay}  (all iterations)"
#         )

#         ax.hist(intra_vis, bins=bins, alpha=0.45, label="intra IAT", color="steelblue")
#         ax.hist(inter_vis, bins=bins, alpha=0.45, label="inter IAT", color="orange")

#         # Reference lines matching the IAT classification plot.
#         ax.axvline(_vis_threshold,   color="red",    linestyle="--", label=f"threshold={_vis_threshold:.0f} μs")
#         ax.axvline(_vis_theoretical, color="green",  linestyle=":",  label=f"theoretical={_vis_theoretical:.0f} μs")
#         ax.axvline(_vis_centers[0],  color="blue",   linestyle=":",  label=f"intra center={_vis_centers[0]:.0f} μs")
#         ax.axvline(_vis_centers[1],  color="orange", linestyle=":",  label=f"inter center={_vis_centers[1]:.0f} μs")

#         fdf = all_dfs.get(fname, pd.DataFrame())
#         if not fdf.empty:
#             if fname in ("real", "ideal"):
#                 best_alpha = None
#                 subset = fdf[
#                     (fdf["rate"] == _vis_rate)
#                     & (fdf["bw"] == _vis_bw)
#                     & (fdf["delay"] == _vis_delay)
#                 ]
#             else:
#                 best_alpha = (
#                     hit_table(fdf, ["formula", "alpha"])
#                     .sort_values("good_rate", ascending=False)
#                     .iloc[0]["alpha"]
#                 )
#                 subset = fdf[
#                     (fdf["alpha"] == best_alpha)
#                     & (fdf["rate"] == _vis_rate)
#                     & (fdf["bw"] == _vis_bw)
#                     & (fdf["delay"] == _vis_delay)
#                 ]

#             if not subset.empty:
#                 ax.hist(
#                     subset["ato"], bins=bins, alpha=0.75,
#                     label=f"ATO  α={best_alpha}", color="crimson",
#                 )
#                 hit_r   = (subset["outcome"] == "hit").mean()
#                 ok_r    = (subset["outcome"] == "ok").mean()
#                 short_r = (subset["outcome"] == "short").mean()
#                 long_r  = (subset["outcome"] == "long").mean()
#                 ax.set_title(
#                     f"{fname}  α={best_alpha}  |  {config.get_rate_name(_vis_rate)}"
#                     f"  bw={_vis_bw}  delay={_vis_delay}\n"
#                     f"hit={hit_r:.0%}  ok={ok_r:.0%}  short={short_r:.0%}  long={long_r:.0%}"
#                 )

#         ax.set_xscale("log")
#         ax.set_xlabel("iat_curr (μs, log scale)")
#         ax.set_ylabel("Count")
#         ax.legend()
#         plt.tight_layout()
#         plt.show()

# ACK send decision

In [ ]:
kernel = 'tcpaad'
reason_rows = []
for rate, bw, delay, iteration in config.conditions:
    ack_chk = dfs[kernel]['ack_chk']
    cond = ((ack_chk['rate'] == rate) & (ack_chk['bw'] == bw)
            & (ack_chk['delay'] == delay) & (ack_chk['iteration'] == iteration))
    sn = ack_chk[cond & (ack_chk['outcome'] == 'SEND_NOW')]

    _m = lambda d: ((d['rate'] == rate) & (d['bw'] == bw)
                    & (d['delay'] == delay) & (d['iteration'] == iteration)).sum()

    row = {
        'rate': rate, 'rate_name': config.get_rate_name(rate),
        'bw': bw, 'delay': delay, 'iteration': iteration,
        'ack_sent': _m(dfs[kernel]['ack_sent']),
        'TIMER': _m(dfs[kernel]['timer_fired']),
        'CLEANUP': _m(dfs[kernel]['cleanup']),
    }
    for flag in _SEND_NOW_FLAGS:
        row[flag] = int(sn[flag].sum()) if flag in sn.columns else 0
    reason_rows.append(row)

df_reasons = pd.DataFrame(reason_rows)
idx = ['rate_name', 'bw', 'delay']
reason_cols = ['TIMER', 'two_seg', 'quick', 'comp_limit', 'CLEANUP', 'ack_now', 'dup_ack']

per_iter = df_reasons.set_index(idx + ['iteration'])
avg = df_reasons.groupby(idx)[['ack_sent'] + reason_cols].mean().round(0).astype(int)
avg.index = pd.MultiIndex.from_tuples(
    [(r, b, d, 'avg') for r, b, d in avg.index], names=idx + ['iteration']
)

combined = pd.concat([per_iter[['ack_sent'] + reason_cols], avg]).sort_index()
for col in reason_cols:
    combined[f'{col}_%'] = (combined[col] / combined['ack_sent'] * 100).round(1)

combined

# RECV event subtypes per config

In [ ]:
kernel = "tcpaad"
recv_df = dfs[kernel]["recv"].copy()
print("=== RECV subtype distribution per config ===\n")
for rate, bw, delay, iteration in config.conditions:
    r = recv_df[
        (recv_df["rate"] == rate)
        & (recv_df["bw"] == bw)
        & (recv_df["delay"] == delay)
        & (recv_df["iteration"] == iteration)
    ]
    if r.empty:
        continue
    total = len(r)
    print(
        f"{config.get_rate_name(rate)} / {delay} bw={bw} iter={iteration} (n={total}):"
    )
    for sub, count in r["subtype"].value_counts().items():
        print(f"  {sub:15s}: {count:>8d}  ({count / total * 100:5.1f}%)")
    print()

# ACK Check outcomes per config

In [ ]:
ack_df = dfs[kernel]['ack_chk'].copy()
ack_df['config'] = ack_df['rate'].map(config.get_rate_name) + ' / ' + ack_df['delay']
print("\n=== ACK_CHK outcome distribution per config ===\n")
for cfg in ack_df['config'].unique():
    r = ack_df[ack_df['config'] == cfg]
    total = len(r)
    print(f"{cfg} (n={total}):")
    for out, count in r['outcome'].value_counts().items():
        print(f"  {out:15s}: {count:>8d}  ({count/total*100:5.1f}%)")
    print()

In [ ]:
ack_pivot = ack_df.groupby(['config', 'outcome']).size().unstack(fill_value=0)
ack_pct = ack_pivot.div(ack_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots()
ack_pct.plot(kind='bar', stacked=True, ax=ax)
ax.set_ylabel('Percentage')
ax.set_title('ACK_CHK outcome distribution per config')
ax.legend(loc='upper right')
ax.set_ylim(0, 100)
plt.tight_layout()
fig.savefig('tmp/ack_chk_outcome_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Event log builder

In [ ]:
def build_event_log(
    dfs,
    config,
    rate,
    bw,
    delay,
    iteration,
    window_start_us,
    window_size_us,
    kernel="tcpaad",
):
    """Build a unified, time-sorted event log for one condition/window."""
    recv = dfs[kernel]["recv"]
    sel = recv[
        (recv["rate"] == rate)
        & (recv["bw"] == bw)
        & (recv["delay"] == delay)
        & (recv["iteration"] == iteration)
    ]
    if sel.empty:
        print(f"Empty: rate={rate} bw={bw} delay={delay} iter={iteration}")
        return pd.DataFrame()

    t0 = sel["ts"].min()
    logs = []
    for ev in EVENTS:
        sub = dfs[kernel][ev]
        sub = sub[
            (sub["rate"] == rate)
            & (sub["bw"] == bw)
            & (sub["delay"] == delay)
            & (sub["iteration"] == iteration)
        ].copy()
        sub["t_rel_us"] = (sub["ts"] - t0).dt.total_seconds() * 1e6
        sub = sub[
            (sub["t_rel_us"] >= window_start_us)
            & (sub["t_rel_us"] < window_start_us + window_size_us)
        ]
        sub["event"] = ev
        logs.append(sub)

    all_logs = (
        pd.concat(logs, ignore_index=True)
        .sort_values("t_rel_us")
        .reset_index(drop=True)
    )

    # Classify each ack_sent by its trigger
    all_logs["ack_trigger"] = ""
    for i in all_logs.index:
        if all_logs.loc[i, "event"] != "ack_sent":
            continue
        ack_rcv_nxt = all_logs.loc[i, "rcv_nxt"]
        for j in range(i - 1, -1, -1):
            row_j = all_logs.loc[j]
            if pd.notna(row_j.get("rcv_nxt")) and row_j["rcv_nxt"] != ack_rcv_nxt:
                continue
            ev = row_j["event"]
            if ev == "ack_chk" and row_j.get("outcome") == "SEND_NOW":
                trigger = next((f for f in _SEND_NOW_FLAGS if row_j.get(f)), "SEND_NOW")
                all_logs.loc[i, "ack_trigger"] = trigger
                break
            elif ev == "timer_ack":
                all_logs.loc[i, "ack_trigger"] = "TIMER"
                break
            elif ev == "cleanup":
                all_logs.loc[i, "ack_trigger"] = "CLEANUP"
                break
            elif ev == "sched":
                st = row_j.get("subtype", "")
                all_logs.loc[i, "ack_trigger"] = (
                    "EARLY_SEND" if st == "EARLY_SEND" else "TIMER_SCHED"
                )
                break
            elif ev == "timer_fired":
                all_logs.loc[i, "ack_trigger"] = "TIMER_FIRED"
                break

    return all_logs

# Trace for specific config

In [ ]:
WINDOW_START_US = 0.03 * 1e6
WINDOW_SIZE_US = 0.02 * 1e6
RATE, BW, DELAY, ITER = 71, 'nolim', 'nodelay', 3

all_logs = build_event_log(dfs, config, RATE, BW, DELAY, ITER,
                            WINDOW_START_US, WINDOW_SIZE_US, kernel='tcpaad')

if not all_logs.empty:
    recv_win = all_logs[all_logs['event'] == 'recv']
    normal = recv_win[recv_win['subtype'] == 'UPDATE']
    noise_win = recv_win[recv_win['subtype'] == 'NOISE']
    stall_win = recv_win[recv_win['subtype'] == 'STALL']
    iat_reset_win = recv_win[recv_win['subtype'] == 'IAT_RESET']
    
    ack_win = all_logs[all_logs['event'] == 'ack_sent'].copy()
    ack_win['delta_bytes'] = ack_win['rcv_nxt'].diff()
    
    TRIGGER_COLORS = {
        'TIMER': 'green', 'two_seg': 'red', 'quick': 'orange',
        'ack_now': 'blue', 'comp_limit': 'brown', 'dup_ack': 'gray',
        'CLEANUP': 'magenta', 'EARLY_SEND': 'cyan', 'TIMER_FIRED': 'darkgreen',
        'TIMER_SCHED': 'lime',
    }
    
    # Compute segment size from sequence numbers (no rcv_mss in kernel logs)
    mss = int((normal['end_seq'] - normal['seq']).median()) if not normal.empty else 1448
    
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.plot(normal['t_rel_us'], normal['iat_curr'], label='UPDATE iat_curr')
    ax.plot(normal['t_rel_us'], normal['iat_min'], label='iat_min')
    if not noise_win.empty:
        ax.plot(noise_win['t_rel_us'], noise_win['iat'], label='NOISE')
    # if not normal.empty:
    #     ax.plot(normal['t_rel_us'], normal['iat_min'], label='noise threshold (iat_min/4)')
    if not stall_win.empty:
        ax.scatter(stall_win['t_rel_us'], stall_win['iat'], label='STALL', marker='x')
    if not iat_reset_win.empty:
        ax.scatter(iat_reset_win['t_rel_us'], iat_reset_win['old_iat_min'], label='IAT_RESET', marker='d')
    
    plotted_triggers = set()
    for _, row in ack_win.iterrows():
        trig = row['ack_trigger'] or 'UNKNOWN'
        for flag, color in TRIGGER_COLORS.items():
            if flag in trig:
                break
            else:
                color = 'black'
      
        label = trig if trig not in plotted_triggers else None
        plotted_triggers.add(trig)
    
        ax.axvline(row['t_rel_us'], color=color, linestyle='--', label=label)
        if pd.notna(row['delta_bytes']) and row['delta_bytes'] > 0:
            segs = int(row['delta_bytes'] / mss)
            ax.text(row['t_rel_us'], ax.get_ylim()[1] * 0.9, str(segs),
                    fontsize=8, rotation=90, color=color, va='top')
    ax.set_yscale('log')
    ax.set_xlabel('Time (us)')
    ax.set_ylabel('IAT (us)')
    ax.set_title(f'{config.get_rate_name(RATE)} | bw={BW} delay={DELAY} iter={ITER}\n'
                 f'window=[{WINDOW_START_US/1e6:.2f}s, {(WINDOW_START_US+WINDOW_SIZE_US)/1e6:.2f}s]')
    ax.legend()
    plt.tight_layout()
    fig.savefig(f'tmp/event_log_{config.get_rate_name(RATE)}_bw={BW}_delay={DELAY}_iter={ITER}'
                f'_window_{WINDOW_START_US/1e6:.2f}s_{(WINDOW_START_US+WINDOW_SIZE_US)/1e6:.2f}s.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## Trace log for specific config

In [ ]:
Path('tmp').mkdir(exist_ok=True)
all_logs.to_csv(
    f'tmp/event_log_{config.get_rate_name(RATE)}_bw={BW}_delay={DELAY}_iter={ITER}'
    f'_window_{WINDOW_START_US/1e6:.2f}s_{(WINDOW_START_US+WINDOW_SIZE_US)/1e6:.2f}s.csv',
    index=False,
)
from IPython.display import display as ipy_display, HTML
html = all_logs.to_html()
ipy_display(HTML(f'<div style="font-size:10px; overflow-x:auto">{html}</div>'))

# IAT_RESET analysis

In [ ]:
df_recv = dfs["tcpaad"]["recv"]
for rate, bw, delay, iteration in config.conditions:
    sel = df_recv[
        (df_recv["rate"] == rate)
        & (df_recv["bw"] == bw)
        & (df_recv["delay"] == delay)
        & (df_recv["iteration"] == iteration)
    ]
    if sel.empty:
        continue
    normal = sel[sel["subtype"] == "UPDATE"]
    if normal.empty:
        continue

    seg_size = (normal["end_seq"] - normal["seq"]).median()
    expected_us = seg_size * 8 / config.phy_rates[rate]
    actual_min = normal["iat_min"].min()
    median_min = normal["iat_min"].median()

    print(
        f"{config.get_rate_name(rate):20s} bw={bw:5s} delay={delay:7s} iter={iteration} | "
        f"expected={expected_us:7.0f}us  iat_min_min={actual_min:5.0f}us  "
        f"iat_min_median={median_min:7.0f}us  ratio={median_min / expected_us:.2f}"
    )

In [ ]:
df_recv = dfs["tcpaad"]["recv"]
iat_rows = []
for rate, bw, delay, iteration in config.conditions:
    sel = df_recv[
        (df_recv["rate"] == rate)
        & (df_recv["bw"] == bw)
        & (df_recv["delay"] == delay)
        & (df_recv["iteration"] == iteration)
    ]
    if sel.empty:
        continue
    normal = sel[sel["subtype"] == "UPDATE"]
    if normal.empty:
        continue
    seg_size = (normal["end_seq"] - normal["seq"]).median()
    expected_us = seg_size * 8 / config.phy_rates[rate]
    median_min = normal["iat_min"].median()
    iat_rows.append(
        {
            "config": f"{config.get_rate_name(rate)} / {delay}",
            "expected_us": expected_us,
            "iat_min_median": median_min,
            "ratio": median_min / expected_us,
            "iteration": iteration,
        }
    )

ratio_df = pd.DataFrame(iat_rows)
avg = (
    ratio_df.groupby("config")
    .agg(
        expected_us=("expected_us", "mean"),
        iat_min_median=("iat_min_median", "mean"),
        ratio=("ratio", "mean"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(avg))
w = 0.35
ax.bar(x - w / 2, avg["expected_us"], w, label="Expected IAT (PHY rate)")
ax.bar(x + w / 2, avg["iat_min_median"], w, label="Measured iat_min (median)")
ax.set_xticks(x)
ax.set_xticklabels(avg["config"], rotation=45, ha="right")
ax.set_ylabel("IAT (us) log")
ax.set_title("Expected vs measured iat_min per configuration")
ax.set_yscale("log")
ax.legend()
for i, r in avg.iterrows():
    ax.text(
        i + w / 2,
        r["iat_min_median"] + 20,
        f"{r['ratio']:.2f}",
        ha="center",
        fontsize=8,
    )
plt.tight_layout()
fig.savefig("tmp/iat_min_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_recv = dfs["tcpaad"]["recv"]
for rate, bw, delay, iteration in config.conditions:
    sel = df_recv[
        (df_recv["rate"] == rate)
        & (df_recv["bw"] == bw)
        & (df_recv["delay"] == delay)
        & (df_recv["iteration"] == iteration)
    ]
    if sel.empty:
        continue
    normal = sel[sel["subtype"] == "UPDATE"]
    if normal.empty:
        continue

    test_duration_us = normal["now_us"].max() - normal["now_us"].min()
    tail_start_us = normal["now_us"].min() + (test_duration_us - 2e6)
    tail = normal[normal["now_us"] >= tail_start_us]

    if not tail.empty:
        print(
            f"{config.get_rate_name(rate):20s} iter={iteration} | "
            f"steady_state_iat_min={tail['iat_min'].max():5.0f}us"
        )

# Debug Test: TCP-AAD vs Default Kernel

Compare the four metrics (throughput, retransmissions, ACK count, RTT) between TCP-AAD and default kernel across all debug test configs.

In [ ]:
perf_rows = []
for kernel in config.kernels:
    for rate, bw, delay, iteration in config.conditions:
        jf = config.raw_path(kernel, rate, bw, delay, iteration)
        if not jf.exists():
            continue
        with open(jf) as f:
            d = json.load(f)

        try:
            sent = d["end"]["sum_sent"]
            stream = d["end"]["streams"][0].get("sender", {})
        except (KeyError, IndexError) as e:
            raise RuntimeError(f"Malformed iperf3 JSON in {jf}: {e}")

        perf_rows.append(
            {
                "kernel": kernel,
                "rate": rate,
                "rate_name": config.get_rate_name(rate),
                "bw": bw,
                "delay": delay,
                "iteration": iteration,
                "tput_mbps": sent["bits_per_second"] / 1e6,
                "retransmits": sent.get("retransmits", 0),
                "mean_rtt_us": stream.get("mean_rtt", 0),
                "min_rtt_us": stream.get("min_rtt", 0),
                "max_rtt_us": stream.get("max_rtt", 0),
            }
        )

df_debug = pd.DataFrame(perf_rows)
df_debug

## Throughput (Mbps)

In [ ]:
idx = ["rate_name", "bw", "delay"]

tput = df_debug.pivot_table(
    index=idx + ["iteration"], columns="kernel", values="tput_mbps"
).round(2)
tput["delta_%"] = ((tput["tcpaad"] - tput["default"]) / tput["default"] * 100).round(2)

tput_avg = df_debug.pivot_table(
    index=idx, columns="kernel", values="tput_mbps", aggfunc="mean"
).round(2)
tput_avg["delta_%"] = (
    (tput_avg["tcpaad"] - tput_avg["default"]) / tput_avg["default"] * 100
).round(2)
tput_avg.index = pd.MultiIndex.from_tuples(
    [(r, b, d, "avg") for r, b, d in tput_avg.index], names=idx + ["iteration"]
)

pd.concat([tput, tput_avg]).sort_index()

## Retransmissions

In [ ]:
retx = (
    df_debug.pivot_table(
        index=idx + ["iteration"], columns="kernel", values="retransmits"
    )
    .round(0)
    .astype(int)
)
retx["delta"] = retx["tcpaad"] - retx["default"]

retx_avg = df_debug.pivot_table(
    index=idx, columns="kernel", values="retransmits", aggfunc="mean"
).round(1)
retx_avg["delta"] = (retx_avg["tcpaad"] - retx_avg["default"]).round(1)
retx_avg.index = pd.MultiIndex.from_tuples(
    [(r, b, d, "avg") for r, b, d in retx_avg.index], names=idx + ["iteration"]
)

pd.concat([retx, retx_avg]).sort_index()

## Mean RTT (µs)

In [ ]:
rtt = (
    df_debug.pivot_table(
        index=idx + ["iteration"], columns="kernel", values="mean_rtt_us"
    )
    .round(0)
    .astype(int)
)
rtt["delta_us"] = rtt["tcpaad"] - rtt["default"]

rtt_avg = df_debug.pivot_table(
    index=idx, columns="kernel", values="mean_rtt_us", aggfunc="mean"
).round(0)
rtt_avg["delta_us"] = (rtt_avg["tcpaad"] - rtt_avg["default"]).round(0)
rtt_avg.index = pd.MultiIndex.from_tuples(
    [(r, b, d, "avg") for r, b, d in rtt_avg.index], names=idx + ["iteration"]
)

pd.concat([rtt, rtt_avg]).sort_index()

## ACK Count (tcpdump)

In [ ]:
idx = ["rate_name", "bw", "delay"]
_ITER_HDR = re.compile(r"^--- Iteration (\d+)")
_ACK_COUNT = re.compile(r"ACK count\s*:\s*(\d+)")

ack_rows = []
for kernel in config.kernels:
    for rate, cases in config.test_plan.items():
        for bw, delay in cases:
            path = config.debug_log_path(kernel, rate, bw, delay)
            if not path.exists():
                continue
            cur_iter = None
            for line in path.read_text().splitlines():
                m = _ITER_HDR.search(line)
                if m:
                    cur_iter = int(m.group(1))
                    continue
                m = _ACK_COUNT.search(line)
                if m and cur_iter is not None:
                    ack_rows.append(
                        {
                            "kernel": kernel,
                            "rate": rate,
                            "rate_name": config.get_rate_name(rate),
                            "bw": bw,
                            "delay": delay,
                            "iteration": cur_iter,
                            "ack_count": int(m.group(1)),
                        }
                    )

df_acks = pd.DataFrame(ack_rows)
if df_acks.empty:
    print("No ACK count data found in logs")
else:
    acks = df_acks.pivot_table(
        index=idx + ["iteration"], columns="kernel", values="ack_count"
    ).astype(int)
    acks["delta"] = acks["tcpaad"] - acks["default"]
    acks["delta_%"] = (
        (acks["tcpaad"] - acks["default"]) / acks["default"] * 100
    ).round(2)

    acks_avg = (
        df_acks.pivot_table(
            index=idx, columns="kernel", values="ack_count", aggfunc="mean"
        )
        .round(0)
        .astype(int)
    )
    acks_avg["delta"] = acks_avg["tcpaad"] - acks_avg["default"]
    acks_avg["delta_%"] = (
        (acks_avg["tcpaad"] - acks_avg["default"]) / acks_avg["default"] * 100
    ).round(2)
    acks_avg.index = pd.MultiIndex.from_tuples(
        [(r, b, d, "avg") for r, b, d in acks_avg.index], names=idx + ["iteration"]
    )

pd.concat([acks, acks_avg]).sort_index()

## Congestion Window (snd_cwnd) over Time

In [ ]:
if __name__ == "__main__":
    import json
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(len(config.rates), len(config.delays), figsize=(14, 3 * len(config.rates)), squeeze=False)

    for col, delay in enumerate(config.delays):
        for row, rate in enumerate(config.rates):
            ax = axes[row][col]
            for kernel in config.kernels:
                jf = config.raw_path(kernel, rate, 'nolim', delay, 1)
                if not jf.exists():
                    continue
                with open(jf) as f:
                    d = json.load(f)
                times = [iv['streams'][0]['end'] for iv in d['intervals']]
                cwnd = [iv['streams'][0]['snd_cwnd'] / 1024 for iv in d['intervals']]  # KB
                ax.plot(times, cwnd, label=kernel)

            ax.set_title(f'{config.get_rate_name(rate)} | delay={delay}')
            ax.set_ylabel('snd_cwnd (KB)')
            ax.legend()
            ax.grid(True, alpha=0.3)

    axes[-1][0].set_xlabel('Time (s)')
    axes[-1][-1].set_xlabel('Time (s)')
    fig.tight_layout()
    fig.savefig('tmp/debug_cwnd.png', dpi=150, bbox_inches='tight')
    plt.show()

## Send Window (snd_wnd) over Time

In [ ]:
if __name__ == "__main__":
    fig, axes = plt.subplots(len(config.rates), len(config.delays), figsize=(14, 3 * len(config.rates)), squeeze=False)

    for col, delay in enumerate(config.delays):
        for row, rate in enumerate(config.rates):
            ax = axes[row][col]
            for kernel in config.kernels:
                jf = config.raw_path(kernel, rate, 'nolim', delay, 1)
                if not jf.exists():
                    continue
                with open(jf) as f:
                    d = json.load(f)
                times = [iv['streams'][0]['end'] for iv in d['intervals']]
                snd_wnd = [iv['streams'][0]['snd_wnd'] / 1024 for iv in d['intervals']]  # KB
                ax.plot(times, snd_wnd, label=kernel)

            ax.set_title(f'{config.get_rate_name(rate)} | delay={delay}')
            ax.set_ylabel('snd_wnd (KB)')
            ax.legend()
            ax.grid(True, alpha=0.3)

    axes[-1][0].set_xlabel('Time (s)')
    axes[-1][-1].set_xlabel('Time (s)')
    fig.tight_layout()
    fig.savefig('tmp/debug_snd_wnd.png', dpi=150, bbox_inches='tight')
    plt.show()